In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                             average_precision_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb
data = pd.read_parquet('data/prepared_data.parquet')


In [12]:
import pandas as pd

data['date'] = pd.to_datetime(data[['Year', 'Month', 'Day']])

data_sorted = data.sort_values('date').reset_index(drop=True)

split_i = int(len(data_sorted) * 0.85)
train_data = data_sorted.iloc[:split_i]
test_data  = data_sorted.iloc[split_i:]

print(f"Train: {len(train_data)} eil | {train_data['date'].min().date()} iki {train_data['date'].max().date()}")
print(f"Test:  {len(test_data)} eil | {test_data['date'].min().date()} iki {test_data['date'].max().date()}")
train_data.drop(columns=['date'], inplace=True)
test_data.drop(columns=['date'], inplace=True)

Train: 2312697 eil | 1995-01-03 iki 2018-04-21
Test:  408124 eil | 2018-04-21 iki 2020-02-28


C:\Users\Liveta\AppData\Local\Temp\ipykernel_4416\1485367253.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data.drop(columns=['date'], inplace=True)
C:\Users\Liveta\AppData\Local\Temp\ipykernel_4416\1485367253.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data.drop(columns=['date'], inplace=True)


In [13]:
mask = train_data['Is Fraud?'] == 1
test = train_data[mask].copy()
print(len(test))

mask = test_data['Is Fraud?'] == 1
test = test_data[mask].copy()
print(len(test))

18221
128


In [14]:

x_train = train_data.drop(columns=['Is Fraud?']).copy()
y_train = train_data['Is Fraud?']
x_test = test_data.drop(columns=['Is Fraud?']).copy()
y_test = test_data['Is Fraud?']

In [15]:
under = RandomUnderSampler(sampling_strategy=0.1, random_state=67)
over  = SMOTE(sampling_strategy=0.5, random_state=67)

resample_pipeline = ImbPipeline([
    ('under', under),
    ('over',  over)
])

X_train_bal, y_train_bal = resample_pipeline.fit_resample(x_train, y_train)
print(f"\nPo balansavimo: {pd.Series(y_train_bal).value_counts().to_dict()}")

X_train_bal = x_train
y_train_bal = y_train


Po balansavimo: {0: 182210, 1: 91105}


In [16]:
percent_fraud = y_train_bal.mean() * 100
print(f"train fraud: {percent_fraud:.2f}%")

percent_fraud = y_test.mean() * 100
print(f"test fraud: {percent_fraud:.2f}%")

train fraud: 0.79%
test fraud: 0.03%


In [17]:

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(X_train_bal)
x_test_scaled = scaler.transform(x_test)


In [18]:
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=67,
    solver='lbfgs'
)
lr.fit(x_train_scaled, y_train_bal)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,67
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [ ]:
fraud_ratio = (y_train_bal == 0).sum() / (y_train_bal == 1).sum()

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=fraud_ratio,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='aucpr',  
    random_state=42,
    use_label_encoder=False
)
xgb_model.fit(
    x_train_scaled, y_train_bal,
    eval_set=[(x_test_scaled, y_test)],
    verbose=50
)

[0]	validation_0-aucpr:0.00075


c:\Users\Liveta\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:54:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[50]	validation_0-aucpr:0.03480
[100]	validation_0-aucpr:0.01282
[150]	validation_0-aucpr:0.01530
[200]	validation_0-aucpr:0.03395
[250]	validation_0-aucpr:0.07393
[299]	validation_0-aucpr:0.07099


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'aucpr'


In [20]:
def evaluate(name, model, X, y):
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y, y_pred, target_names=['Ne-fraud', 'Fraud']))
    print(f"ROC-AUC:  {roc_auc_score(y, y_proba):.4f}")
    print(f"PR-AUC:   {average_precision_score(y, y_proba):.4f}")
    print(f"Confusion matrix:\n{confusion_matrix(y, y_pred)}")

evaluate("Logistic Regression", lr,        x_test_scaled, y_test)
evaluate("XGBoost",             xgb_model, x_test_scaled, y_test)


  Logistic Regression
              precision    recall  f1-score   support

    Ne-fraud       1.00      0.68      0.81    407996
       Fraud       0.00      0.66      0.00       128

    accuracy                           0.68    408124
   macro avg       0.50      0.67      0.40    408124
weighted avg       1.00      0.68      0.81    408124

ROC-AUC:  0.7219
PR-AUC:   0.0013
Confusion matrix:
[[276260 131736]
 [    43     85]]

  XGBoost
              precision    recall  f1-score   support

    Ne-fraud       1.00      1.00      1.00    407996
       Fraud       0.00      0.00      0.00       128

    accuracy                           1.00    408124
   macro avg       0.50      0.50      0.50    408124
weighted avg       1.00      1.00      1.00    408124

ROC-AUC:  0.8676


c:\Users\Liveta\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Liveta\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Liveta\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

PR-AUC:   0.0752
Confusion matrix:
[[407996      0]
 [   128      0]]
